In [1]:
import pandas as pd

true_df=pd.read_csv("True.csv")
fake_df=pd.read_csv("Fake.csv")
true_df['label']='REAL'
fake_df['label']='FAKE'

In [2]:
real_sample=true_df.sample(n=2000, random_state=42)
fake_sample=fake_df.sample(n=2000, random_state=42)

reduced_df=pd.concat([real_sample, fake_sample]).sample(frac=1, random_state=42).reset_index(drop=True)
print(reduced_df['label'].value_counts())
print(reduced_df.head())

label
REAL    2000
FAKE    2000
Name: count, dtype: int64
                                               title  \
0  May's government pushes Brexit bill to avoid '...   
1   Trump’s EPA OKs Pesticide That Causes Brain D...   
2  Man arrested at Trump rally said he wanted to ...   
3  Jared Kushner NEVER Registered To Vote As A “F...   
4  MARTHA STEWART Makes Lewd Gesture Towards Trum...   

                                                text       subject  \
0  LONDON (Reuters) - Brexit minister David Davis...     worldnews   
1  Farmworkers were pulled from fields on Friday ...          News   
2  (Reuters) - A man arrested over the weekend tr...  politicsNews   
3  Meanwhile, as President Trump continues to mee...     left-news   
4  Martha, Martha, Martha You re 75-years old! Ti...     left-news   

                 date label  
0  September 6, 2017   REAL  
1        May 15, 2017  FAKE  
2      June 20, 2016   REAL  
3        Sep 29, 2017  FAKE  
4         May 8, 2017  FAKE  


In [3]:
reduced_df['content']=reduced_df['title']+ " "+reduced_df['text']
reduced_df=reduced_df[['content', 'label']]

In [4]:
from sklearn.preprocessing import LabelEncoder

label=LabelEncoder()
y_encoded=label.fit_transform(reduced_df['label'])
print(label.classes_)

['FAKE' 'REAL']


In [5]:
import re
import nltk
nltk.download ('stopwords')

from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Madhurjya\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [6]:
stop_words=set(stopwords.words('english'))
def clean_text(text):
    text=text.lower()
    text=re.sub(r'[^a-z\s]', '', text)
    words=text.split()
    words=[w for w in words if w not in stop_words]
    return " ".join(words)
reduced_df['clean_text']=reduced_df['content'].apply(clean_text)


In [7]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words=1000
max_len=300

token=Tokenizer(num_words=max_words, oov_token="<00V>")
token.fit_on_texts(reduced_df['clean_text'])
sequences=token.texts_to_sequences(reduced_df['clean_text'])
pad=pad_sequences(sequences, maxlen=max_len, padding='post')

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test=train_test_split(pad, y_encoded, test_size=0.2, random_state=42)

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

In [15]:
model_bilstm = Sequential()

# Word embeddings
model_bilstm.add(
    Embedding(
        input_dim=max_words,
        output_dim=128,
        input_length=max_len
    )
)

# Bidirectional LSTM
model_bilstm.add(
    Bidirectional(
        LSTM(64, return_sequences=False)
    )
)

# Regularization
model_bilstm.add(Dropout(0.5))

# Dense layer
model_bilstm.add(Dense(64, activation='relu'))

# Output layer
model_bilstm.add(Dense(1, activation='sigmoid'))

model_bilstm.build(input_shape=(None, max_len))
model_bilstm.summary()

c:\Users\Madhurjya\Desktop\b drive\projectss\fake_news\fakenews\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 300, 128)       │       128,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 235,137 (918.50 KB)

 Trainable params: 235,137 (918.50 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
model_bilstm.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [12]:
history_bilstm = model_bilstm.fit(X_train, y_train, epochs=5, batch_size=32, validation_split=0.2)

Epoch 1/5
80/80 ━━━━━━━━━━━━━━━━━━━━ 10s 87ms/step - accuracy: 0.7449 - loss: 0.4977 - val_accuracy: 0.9375 - val_loss: 0.1463
Epoch 2/5
80/80 ━━━━━━━━━━━━━━━━━━━━ 7s 85ms/step - accuracy: 0.9539 - loss: 0.1133 - val_accuracy: 0.9531 - val_loss: 0.1098
Epoch 3/5
80/80 ━━━━━━━━━━━━━━━━━━━━ 7s 85ms/step - accuracy: 0.9859 - loss: 0.0522 - val_accuracy: 0.9766 - val_loss: 0.0733
Epoch 4/5
80/80 ━━━━━━━━━━━━━━━━━━━━ 8s 96ms/step - accuracy: 0.9914 - loss: 0.0261 - val_accuracy: 0.9594 - val_loss: 0.1139
Epoch 5/5
80/80 ━━━━━━━━━━━━━━━━━━━━ 8s 96ms/step - accuracy: 0.9902 - loss: 0.0378 - val_accuracy: 0.9812 - val_loss: 0.0556


In [13]:
test_loss, test_accuracy = model_bilstm.evaluate(X_test, y_test)

print("BiLSTM Test Accuracy:", test_accuracy)

25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.9787 - loss: 0.0714
BiLSTM Test Accuracy: 0.9787499904632568


In [14]:
import numpy as np
from sklearn.metrics import confusion_matrix

y_pred = (model_bilstm.predict(X_test) > 0.5).astype(int)

cm = confusion_matrix(y_test, y_pred)

print(cm)

25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step
[[387  15]
 [  2 396]]
